In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("SmartGrid") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(spark.version)

4.0.2


In [ ]:
import sys
import os
import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn for ML models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Visualization
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns

# Set style for publication-quality figures
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10


def load_data_pandas(data_path):
    """Load data efficiently with pandas."""
    print(f"\nLoading data from: {data_path}")
    start_time = time.time()

    # Read with pandas
    df = pd.read_csv(
        data_path,
        sep=';',
        header=0,
        na_values=['?'],
        dtype={
            'Global_active_power': 'float64',
            'Global_reactive_power': 'float64',
            'Voltage': 'float64',
            'Global_intensity': 'float64',
            'Sub_metering_1': 'float64',
            'Sub_metering_2': 'float64',
            'Sub_metering_3': 'float64'
        },
        low_memory=False
    )

    # Rename columns
    df.columns = [
        'Date', 'Time', 'Global_active_power', 'Global_reactive_power',
        'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
    ]

 # Create datetime
    df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='mixed', dayfirst=True)

    # Drop NaN (حذف القيم الفارغة أو التي فشل تحويل تاريخها)
    df = df.dropna(subset=['Global_active_power', 'DateTime'])

    load_time = time.time() - start_time
    print(f"  Total rows loaded: {len(df):,}")
    print(f"  Load time: {load_time:.2f} seconds")

    return df

def create_features(df):
    """Create features efficiently."""
    print("\nPerforming feature engineering...")
    start_time = time.time()

    df = df.copy()

    # Time features
    df['hour'] = df['DateTime'].dt.hour
    df['day_of_week'] = df['DateTime'].dt.dayofweek
    df['month'] = df['DateTime'].dt.month

    # Cyclical encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    # Sort for lag features
    df = df.sort_values('DateTime').reset_index(drop=True)

    # Lag features
    for lag in [1, 3, 6, 24]:
        df[f'lag_{lag}h_active_power'] = df['Global_active_power'].shift(lag)
        df[f'lag_{lag}h_sub_1'] = df['Sub_metering_1'].shift(lag)

    # Rolling features
    for window in [3, 6, 24]:
        df[f'rolling_mean_{window}h'] = df['Global_active_power'].rolling(window=window, min_periods=1).mean()

    # Derived features
    df['total_sub_metering'] = df['Sub_metering_1'] + df['Sub_metering_2'] + df['Sub_metering_3']

    elapsed = time.time() - start_time
    print(f"  Feature engineering complete: {elapsed:.2f} seconds")

    # Feature columns
    feature_cols = [
        'Global_reactive_power', 'Voltage', 'Global_intensity',
        'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend',
        'lag_1h_active_power', 'lag_3h_active_power', 'lag_6h_active_power', 'lag_24h_active_power',
        'lag_1h_sub_1', 'lag_3h_sub_1',
        'rolling_mean_3h', 'rolling_mean_6h', 'rolling_mean_24h',
        'total_sub_metering'
    ]

    # Drop NaN
    df_clean = df.dropna(subset=feature_cols + ['Global_active_power'])
    print(f"  Final samples: {len(df_clean):,}")
    print(f"  Total features: {len(feature_cols)}")

    return df_clean, feature_cols


def train_and_evaluate_models(X_train, X_test, y_train, y_test):
    """Train and evaluate all models."""
    results = {}

    # 1. Linear Regression
    print("\n" + "="*60)
    print("TRAINING: Linear Regression")
    print("="*60)

    start = time.time()
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    train_time = time.time() - start

    start = time.time()
    y_pred = lr.predict(X_test)
    pred_time = time.time() - start

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    results['LinearRegression'] = {
        'training_time_sec': train_time,
        'prediction_time_sec': pred_time,
        'metrics': {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': np.mean(np.abs((y_test - y_pred) / y_test)) * 100}
    }
    print(f"  Time: {train_time:.2f}s, RMSE: {rmse:.4f}, R²: {r2:.4f}")

    # 2. Random Forest
    print("\n" + "="*60)
    print("TRAINING: Random Forest")
    print("="*60)

    start = time.time()
    rf = RandomForestRegressor(n_estimators=30, max_depth=10, n_jobs=-1, random_state=42)
    rf.fit(X_train, y_train)
    train_time = time.time() - start

    start = time.time()
    y_pred_rf = rf.predict(X_test)
    pred_time = time.time() - start

    rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
    mae = mean_absolute_error(y_test, y_pred_rf)
    r2 = r2_score(y_test, y_pred_rf)

    results['RandomForest'] = {
        'training_time_sec': train_time,
        'prediction_time_sec': pred_time,
        'metrics': {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100},
        'feature_importance': rf.feature_importances_[:10].tolist()
    }
    print(f"  Time: {train_time:.2f}s, RMSE: {rmse:.4f}, R²: {r2:.4f}")

    # 3. Gradient Boosting
    print("\n" + "="*60)
    print("TRAINING: Gradient Boosting")
    print("="*60)

    start = time.time()
    gb = GradientBoostingRegressor(n_estimators=50, max_depth=5, learning_rate=0.1, random_state=42)
    gb.fit(X_train, y_train)
    train_time = time.time() - start

    start = time.time()
    y_pred_gb = gb.predict(X_test)
    pred_time = time.time() - start

    rmse = np.sqrt(mean_squared_error(y_test, y_pred_gb))
    mae = mean_absolute_error(y_test, y_pred_gb)
    r2 = r2_score(y_test, y_pred_gb)

    results['GradientBoosting'] = {
        'training_time_sec': train_time,
        'prediction_time_sec': pred_time,
        'metrics': {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': np.mean(np.abs((y_test - y_pred_gb) / y_test)) * 100},
        'feature_importance': gb.feature_importances_[:10].tolist()
    }
    print(f"  Time: {train_time:.2f}s, RMSE: {rmse:.4f}, R²: {r2:.4f}")

    return results, {'LinearRegression': y_pred, 'RandomForest': y_pred_rf, 'GradientBoosting': y_pred_gb}


def run_scalability_analysis(X_train, y_train, X_test, y_test):
    """Run scalability analysis."""
    print("\n" + "="*70)
    print("SCALABILITY ANALYSIS")
    print("="*70)

    scalability = {}

    # Data size scaling
    print("\n--- Data Size Scaling ---")
    data_scaling = {}
    fractions = [0.25, 0.5, 0.75, 1.0]
    baseline_time = None
    baseline_size = None

    for frac in fractions:
        n_samples = int(len(X_train) * frac)
        X_sample = X_train[:n_samples]
        y_sample = y_train[:n_samples]

        start = time.time()
        model = LinearRegression()
        model.fit(X_sample, y_sample)
        train_time = time.time() - start

        if baseline_time is None:
            baseline_time = train_time
            baseline_size = n_samples
            factor = 1.0
        else:
            size_ratio = n_samples / baseline_size
            time_ratio = train_time / baseline_time
            factor = time_ratio / size_ratio

        data_scaling[str(frac)] = {
            'fraction': frac,
            'train_samples': n_samples,
            'training_time_sec': train_time,
            'scaling_factor': factor
        }
        print(f"  {frac*100:.0f}% ({n_samples:,} samples): {train_time:.2f}s (factor: {factor:.2f})")

    scalability['data_scaling'] = data_scaling

    # Model complexity scaling (fast configs)
    print("\n--- Model Complexity Scaling ---")
    complexity = []
    configs = [
        {'n_estimators': 10, 'max_depth': 5},
        {'n_estimators': 30, 'max_depth': 10},
    ]

    for config in configs:
        start = time.time()
        model = RandomForestRegressor(**config, n_jobs=-1, random_state=42)
        model.fit(X_train, y_train)
        train_time = time.time() - start

        y_pred = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))

        complexity.append({'config': config, 'training_time_sec': train_time, 'rmse': rmse})
        print(f"  RF {config}: {train_time:.2f}s, RMSE: {rmse:.4f}")

    scalability['model_complexity'] = {'RandomForest': complexity}

    return scalability


def generate_figures(df_sample, results, scalability, output_dir):
    """Generate all figures."""
    print("\n" + "="*70)
    print("GENERATING VISUALIZATIONS")
    print("="*70)

    figures_dir = Path(output_dir)
    figures_dir.mkdir(parents=True, exist_ok=True)

    # Figure 1: Dataset characteristics
    print("\nGenerating Figure 1...")
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Daily
    daily_avg = df_sample.groupby(df_sample['DateTime'].dt.hour)['Global_active_power'].mean()
    axes[0, 0].plot(daily_avg.index, daily_avg.values, linewidth=2, color='#2E86AB')
    axes[0, 0].set_xlabel('Hour of Day')
    axes[0, 0].set_ylabel('Average Power (kW)')
    axes[0, 0].set_title('(a) Daily Energy Consumption Pattern')
    axes[0, 0].grid(True, alpha=0.3)

    # Weekly
    weekly_avg = df_sample.groupby(df_sample['DateTime'].dt.dayofweek)['Global_active_power'].mean()
    axes[0, 1].bar(range(7), weekly_avg.values, color='#A23B72', alpha=0.8)
    axes[0, 1].set_xlabel('Day of Week')
    axes[0, 1].set_ylabel('Average Power (kW)')
    axes[0, 1].set_title('(b) Weekly Pattern')
    axes[0, 1].set_xticks(range(7))
    axes[0, 1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    axes[0, 1].grid(True, alpha=0.3, axis='y')

    # Monthly
    monthly_avg = df_sample.groupby(df_sample['DateTime'].dt.month)['Global_active_power'].mean()
    axes[1, 0].bar(range(1, 13), monthly_avg.values, color='#F18F01', alpha=0.8)
    axes[1, 0].set_xlabel('Month')
    axes[1, 0].set_ylabel('Average Power (kW)')
    axes[1, 0].set_title('(c) Monthly Pattern')
    axes[1, 0].grid(True, alpha=0.3, axis='y')

    # Distribution
    axes[1, 1].hist(df_sample['Global_active_power'], bins=50, color='#C73E1D', alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Global Active Power (kW)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('(d) Distribution')
    axes[1, 1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(figures_dir / 'fig1_dataset_characteristics.png', bbox_inches='tight', facecolor='white')
    plt.close()
    print("  Saved: fig1_dataset_characteristics.png")

    # Figure 2: Model performance
    print("\nGenerating Figure 2...")
    models = list(results.keys())
    rmse = [results[m]['metrics']['RMSE'] for m in models]
    mae = [results[m]['metrics']['MAE'] for m in models]
    r2 = [results[m]['metrics']['R2'] for m in models]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    colors = ['#2E86AB', '#A23B72', '#F18F01']

    for ax, metric, values, title in zip(axes, ['RMSE', 'MAE', 'R²'], [rmse, mae, r2],
                                          ['(a) Root Mean Square Error', '(b) Mean Absolute Error', '(c) R² Score']):
        bars = ax.bar(models, values, color=colors, alpha=0.8, edgecolor='black')
        ax.set_ylabel(f'{metric} (kW)' if metric != 'R²' else metric)
        ax.set_title(title)
        ax.grid(True, alpha=0.3, axis='y')
        for i, v in enumerate(values):
            ax.text(i, v + max(values)*0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
        ax.tick_params(axis='x', rotation=15)
        if metric == 'R²':
            ax.set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig(figures_dir / 'fig2_model_performance.png', bbox_inches='tight', facecolor='white')
    plt.close()
    print("  Saved: fig2_model_performance.png")

    # Figure 3: Training time
    print("\nGenerating Figure 3...")
    train_times = [results[m]['training_time_sec'] for m in models]
    pred_times = [results[m]['prediction_time_sec'] for m in models]

    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(models))
    width = 0.35

    ax.bar(x - width/2, train_times, width, label='Training', color='#2E86AB', alpha=0.8, edgecolor='black')
    ax.bar(x + width/2, pred_times, width, label='Prediction', color='#A23B72', alpha=0.8, edgecolor='black')
    ax.set_ylabel('Time (seconds)')
    ax.set_title('Training and Prediction Time Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=15, ha='right')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(figures_dir / 'fig3_training_time.png', bbox_inches='tight', facecolor='white')
    plt.close()
    print("  Saved: fig3_training_time.png")

    # Figure 4: Data scaling
    print("\nGenerating Figure 4...")
    data_scaling = scalability['data_scaling']
    fractions = [data_scaling[k]['fraction'] for k in data_scaling.keys()]
    samples = [data_scaling[k]['train_samples'] for k in data_scaling.keys()]
    times = [data_scaling[k]['training_time_sec'] for k in data_scaling.keys()]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(samples, times, 'o-', linewidth=2, markersize=8, color='#F18F01')
    axes[0].set_xlabel('Number of Training Samples')
    axes[0].set_ylabel('Training Time (seconds)')
    axes[0].set_title('(a) Training Time vs. Data Size')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot([f*100 for f in fractions], times, 's-', linewidth=2, markersize=8, color='#C73E1D')
    axes[1].set_xlabel('Data Fraction (%)')
    axes[1].set_ylabel('Training Time (seconds)')
    axes[1].set_title('(b) Training Time Scaling')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(figures_dir / 'fig4_data_scaling.png', bbox_inches='tight', facecolor='white')
    plt.close()
    print("  Saved: fig4_data_scaling.png")

    print("\nAll visualizations complete!")


def main():
    """Main pipeline."""
    overall_start = time.time()

    print("="*70)
    print("SMART GRID ENERGY CONSUMPTION PREDICTION PIPELINE")
    print("Big Data Analytics with Machine Learning")
    print("="*70)

   # Paths لبيئة العمل المحلية أو كولاب
    data_path = "household_power_consumption.txt"  # ضع الملف في نفس المجلد
    output_dir = "./results"                       # سيتم إنشاء مجلد النتائج هنا تلقائياً
    figures_dir = "./figures"                     # سيتم إنشاء مجلد الصور هنا تلقائياً

    Path(output_dir).mkdir(parents=True, exist_ok=True)
    Path(figures_dir).mkdir(parents=True, exist_ok=True)

    # Load data
    print("\n" + "="*70)
    print("PHASE 1: DATA LOADING")
    print("="*70)
    df = load_data_pandas(data_path)

    # Feature engineering
    print("\n" + "="*70)
    print("PHASE 2: FEATURE ENGINEERING")
    print("="*70)
    df_features, feature_cols = create_features(df)

    # Prepare data
    X = df_features[feature_cols].values
    y = df_features['Global_active_power'].values

    print("\n" + "="*70)
    print("PHASE 3: TRAIN-TEST SPLIT")
    print("="*70)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    print(f"Training samples: {len(X_train):,}")
    print(f"Test samples: {len(X_test):,}")

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train models
    print("\n" + "="*70)
    print("PHASE 4: MODEL TRAINING")
    print("="*70)
    results, predictions = train_and_evaluate_models(X_train_scaled, X_test_scaled, y_train, y_test)

    # Save results
    with open(f"{output_dir}/benchmark_results.json", 'w') as f:
        json.dump(results, f, indent=2)

    # Save prediction samples
    for name, y_pred in predictions.items():
        sample = pd.DataFrame({'actual': y_test[:5000], 'predicted': y_pred[:5000]})
        sample.to_csv(f"{output_dir}/{name}_predictions_sample.csv", index=False)

    # Scalability analysis
    scalability = run_scalability_analysis(X_train_scaled, y_train, X_test_scaled, y_test)
    with open(f"{output_dir}/scalability_results.json", 'w') as f:
        json.dump({'scalability_analysis': scalability}, f, indent=2)

    # Generate figures
    df_sample = df_features.sample(n=min(100000, len(df_features)), random_state=42)
    generate_figures(df_sample, results, scalability, figures_dir)

    # Final summary
    overall_time = time.time() - overall_start

    print("\n" + "="*70)
    print("PIPELINE COMPLETE")
    print("="*70)
    print(f"\nTotal execution time: {overall_time:.2f} seconds ({overall_time/60:.2f} minutes)")

    print("\n" + "="*70)
    print("FINAL RESULTS SUMMARY")
    print("="*70)

    print(f"\n{'Model':<25} {'RMSE':<12} {'MAE':<12} {'R²':<10} {'Time (s)':<12}")
    print("-" * 70)
    for model_name, result in results.items():
        metrics = result['metrics']
        print(f"{model_name:<25} "
              f"{metrics['RMSE']:<12.4f} "
              f"{metrics['MAE']:<12.4f} "
              f"{metrics['R2']:<10.4f} "
              f"{result['training_time_sec']:<12.2f}")

    best = min(results.items(), key=lambda x: x[1]['metrics']['RMSE'])
    print(f"\nBest Model (by RMSE): {best[0]}")

    return results


if __name__ == "__main__":
    main()


SMART GRID ENERGY CONSUMPTION PREDICTION PIPELINE
Big Data Analytics with Machine Learning

PHASE 1: DATA LOADING

Loading data from: household_power_consumption.txt
  Total rows loaded: 2,049,280
  Load time: 143.43 seconds

PHASE 2: FEATURE ENGINEERING

Performing feature engineering...
  Feature engineering complete: 1.31 seconds
  Final samples: 2,049,256
  Total features: 23

PHASE 3: TRAIN-TEST SPLIT
Training samples: 1,639,404
Test samples: 409,852

PHASE 4: MODEL TRAINING

TRAINING: Linear Regression
  Time: 1.96s, RMSE: 0.0374, R²: 0.9983

TRAINING: Random Forest
  Time: 347.90s, RMSE: 0.0326, R²: 0.9987

TRAINING: Gradient Boosting
  Time: 609.11s, RMSE: 0.0295, R²: 0.9989

SCALABILITY ANALYSIS

--- Data Size Scaling ---
  25% (409,851 samples): 0.44s (factor: 1.00)
  50% (819,702 samples): 0.99s (factor: 1.12)
  75% (1,229,553 samples): 1.87s (factor: 1.42)
  100% (1,639,404 samples): 2.03s (factor: 1.15)

--- Model Complexity Scaling ---
  RF {'n_estimators': 10, 'max_depth